<a href="https://colab.research.google.com/github/Arfa-Tariq/learning-ai-engineering/blob/main/projects/03-ResumeAnalyzer/resume_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📄 Resume Analyzer & Critiquer

AI-powered resume analysis with extraction and critique against job descriptions.

---

## 📌 Project Overview

This notebook builds an end-to-end resume analysis tool that:

1. Extracts structured information from PDF resumes
2. Compares the candidate profile against a job description
3. Provides a detailed critique with match percentage, strengths, gaps, and recommendations

---

## ⚙️ Workflow

| Step | Action | Output |
|------|--------|--------|
| 1 | User uploads a PDF resume | - |
| 2 | Docling converts PDF to Markdown | Clean, structured text |
| 3 | LLM extracts key information | JSON: name, skills, experience, education |
| 4 | User pastes a job description | - |
| 5 | LLM compares and critiques | Match %, strengths, gaps, recommendations |

---

## 🛠️ Technologies Used

| Technology | Purpose |
|------------|---------|
| **Docling** | PDF to Markdown conversion (handles layouts, tables, OCR) |
| **Groq** | Fast LLM inference (Llama 3.3 70B) |
| **Gradio** | Interactive UI with file upload and text display |
| **Google Colab** | Free environment with secrets management |

---

## 🚀 How to Use

1. Run all cells in this notebook
2. Upload a resume PDF using the Gradio UI
3. Paste a job description in the text box
4. Click **"Analyze Resume"**
5. View extracted information and detailed critique


In [1]:
!pip install -q groq docling gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 9.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 575.4/575.4 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.8/257.8 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 21.8 MB/s eta 0:00:00
   ━━━

In [2]:
#Impoting Libraries

import os, json, tempfile
from docling.document_converter import DocumentConverter
from groq import Groq
from google.colab import userdata
import gradio as gr

In [3]:
#Setting up Groq Client

GROQ_API_KEY = userdata.get('groq_key')
client = Groq(api_key=GROQ_API_KEY)
def get_completion(messages, model="openai/gpt-oss-120b", temperature=0):
  response = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=temperature,)
  return response.choices[0].message.content

In [4]:
#Initialize Docling Covertor

convertor = DocumentConverter()

In [5]:
analysis_context = []

In [6]:
def extract_resume_info(resume_md):
    """
    Extract structured information from resume markdown.
    Uses system role for consistent behavior.
    """
    system_prompt = """
You are a professional resume parser. Your task is to extract structured information from resumes.

Extract the following fields and return them as valid JSON:
1. **name**: Full name of the candidate (string)
2. **contact**: Object with email, phone, location
3. **skills**: Object with:
   - technical: List of technical skills (programming languages, frameworks, tools)
   - soft: List of soft skills (communication, leadership, etc.)
4. **experience**: List of objects, each with:
   - company: Company name
   - role: Job title
   - duration: Start year - End year
   - achievements: List of key achievements (max 3 per role)
5. **education**: List of objects, each with:
   - degree: Degree name
   - institution: University name
   - year: Graduation year
6. **certifications**: List of certifications (empty list if none)
7. **summary**: 2-3 sentence summary of the candidate's profile (max 80 words)

Return ONLY valid JSON with these keys:
name, contact, skills, experience, education, certifications, summary

If a field is not found, use null for strings, empty list for lists, empty object for objects.
Do not add any extra text outside the JSON.
"""

    # System prompt sets the behavior once
    analysis_context.append({'role': 'system', 'content': system_prompt})

    # User prompt contains the actual resume
    user_prompt = f"""
Resume (delimited by triple backticks): ```{resume_md}```
"""
    analysis_context.append({'role': 'user', 'content': user_prompt})
    response = get_completion(analysis_context)
    analysis_context.append({'role': 'assistant', 'content': response})

    try:
        return json.loads(response)
    except:
        return {"error": "Failed to parse JSON", "raw": response}

In [7]:
def critique_resume(extracted_data, job_description):
    """
    Critique resume against job description.
    Uses system role for consistent behavior.
    """
    system_prompt = """
You are a professional resume reviewer and career coach. Your task is to analyze how well a candidate's resume matches a job description.

You must provide:
1. **Match Percentage (0-100%)**: How well does the resume fit this job?
2. **Strengths (3-5 items)**: Highlight the candidate's strong points for this role
3. **Gaps**: Missing skills, experience, or qualifications
4. **Recommendations**: Actionable steps to improve the resume for this role
5. **Final Verdict**: Hire / Strong Consider / Consider / Pass

Format your response with clear headings and bullet points.
Be objective and constructive in your feedback.
"""

    # System prompt sets the behavior once
    analysis_context.append({'role': 'system', 'content': system_prompt})

    # User prompt contains the actual data
    user_prompt = f"""
## Resume Data (JSON):
{json.dumps(extracted_data, indent=2)}

## Job Description:
{job_description}

Provide your analysis:
"""
    analysis_context.append({'role': 'user', 'content': user_prompt})
    response = get_completion(analysis_context)
    analysis_context.append({'role': 'assistant', 'content': response})
    return response

In [8]:
def reset_context():
    """Clear the analysis context for a new session"""
    global analysis_context
    analysis_context = []
    return "Context cleared"

In [10]:
def process_resume(pdf_file, job_description):
    """
    Process resume PDF and critique against job description.
    Both inputs required.
    """
    reset_context()
    try:
        # 1. Save uploaded PDF to temporary file
        with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
            tmp.write(pdf_file)
            tmp_path = tmp.name

        # 2. Convert PDF to Markdown
        result = convertor.convert(tmp_path)
        resume_md = result.document.export_to_markdown()

        # 3. Extract structured information
        resume_data = extract_resume_info(resume_md)

        # 4. Generate critique
        critique = critique_resume(resume_data, job_description)

        # 5. Clean up temporary file
        os.unlink(tmp_path)

        # 6. Return results
        return resume_data, critique

    except Exception as e:
        return {"error": str(e)}, f"❌ Error: {str(e)}"

In [ ]:
with gr.Blocks(
    title="Resume Analyzer",
    theme=gr.themes.Soft(),
    css="""
        .gradio-container {max-width: 1000px; margin: auto;}
        .header {text-align: center; padding: 20px;}
        .header h1 {font-size: 2.5rem; margin-bottom: 0;}
        .header p {font-size: 1.1rem; color: #666;}
        .feedback {padding: 15px; border-radius: 8px;}
    """
) as demo:

    # HEADER

    gr.Markdown("""
    # 📄 Resume Analyzer & Critiquer

    Upload your resume PDF and paste a job description to get a detailed analysis with:
    - **Match Percentage** – How well your resume fits the role
    - **Strengths** – Your strong points for this position
    - **Gaps** – Missing skills or experience
    - **Recommendations** – Actionable steps to improve
    - **Final Verdict** – Hire, Strong Consider, Consider, or Pass

    ---
    """)

    # MAIN LAYOUT

    with gr.Row():
        # LEFT COLUMN – Inputs
        with gr.Column(scale=1):
            gr.Markdown("### 📤 Step 1: Upload Your Resume")
            resume_upload = gr.File(
                label="Upload Resume (PDF)",
                file_types=[".pdf"],
                type="binary"
            )

            gr.Markdown("### 💼 Step 2: Paste Job Description")
            job_description = gr.Textbox(
                label="Job Description",
                placeholder="Paste the full job description here...",
                lines=10
            )

            with gr.Row():
                analyze_btn = gr.Button("🔍 Analyze Resume", variant="primary", size="lg")
                clear_btn = gr.Button("🔄 Clear All", variant="secondary", size="lg")

        # RIGHT COLUMN – Outputs
        with gr.Column(scale=1):
            gr.Markdown("### 📋 Extracted Information")
            extracted_info = gr.JSON(label="", value={})

            gr.Markdown("### 🔍 Resume Critique")
            critique_output = gr.Markdown(
                label="",
                value="👆 Upload a resume and paste a job description, then click **Analyze Resume**."
            )

    # FOOTER
    gr.Markdown("""
    ---
    **Built with:** Docling • Groq (openai/gpt-oss-120b) • Gradio

    *Your resume data is processed in real-time and not stored.*
    """)

    # EVENT HANDLERS

    # Analyze Button
    analyze_btn.click(
        fn=process_resume,
        inputs=[resume_upload, job_description],
        outputs=[extracted_info, critique_output]
    )

    # Clear Button
    def clear_all():
        return None, "", {}, "👆 Upload a resume and paste a job description, then click **Analyze Resume**."

    clear_btn.click(
        fn=clear_all,
        inputs=[],
        outputs=[resume_upload, job_description, extracted_info, critique_output]
    )

# LAUNCH THE APP
demo.launch(share=True, debug=True)